In [8]:
import pandas as pd
import requests

# Step 1: Load the CSV file
file_path = r"C:\Users\Chris\Documents\ALEK\DataSci\Data\Constituition.csv"
df = pd.read_csv(file_path)

# Step 2: Define column names (matching your CSV)
chapter_column = "CHAPTER"
question_column = "QUESTION"
answer_column = "ANSWER"

# Step 3: Check for missing or incorrect data
required_columns = [chapter_column, question_column, answer_column]
for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in the CSV file.")

# Drop rows with missing values
df = df.dropna(subset=required_columns)

# Step 4: Define the ask_ollama function
def ask_ollama(question, model="deepseek-r1:1.5b"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": f"You are a Kenyan civic education expert. Answer concisely and accurately based on the Constitution of Kenya: {question}",
        "stream": False
    }
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.text}"

# Step 5: Define the validate_response function
def validate_response(response, expected_keywords):
    return any(keyword.lower() in response.lower() for keyword in expected_keywords)

# Step 6: Define the clean_response function
def clean_response(response):
    irrelevant_phrases = ["I think", "Hmm", "Let me", "Wait", "But"]
    for phrase in irrelevant_phrases:
        response = response.replace(phrase, "")
    return response.strip()

# Step 7: Define the fallback mechanism
fallback_answers = {
    "What does Article 26 say about life?": "Article 26 states that every person has the right to life. The life of a person begins at conception, and no one shall be deprived of life intentionally except as authorized by the Constitution or other written law.",
    "Explain devolution in Kenya.": "Devolution is the decentralization of power to county governments, as outlined in Chapter 11 of the Kenyan Constitution.",
}

def ask_ollama_with_fallback(question):
    response = ask_ollama(question)
    if not validate_response(response, ["right to life", "Article 26"]):
        return fallback_answers.get(question, "I don't have an answer for that question.")
    return response

# Step 8: Generate answers for all questions
results = []
for index, row in df.iterrows():
    chapter = row[chapter_column]
    question = row[question_column]
    expected_answer = row[answer_column]
    model_answer = ask_ollama_with_fallback(question)
    model_answer = clean_response(model_answer)
    results.append({
        "chapter": chapter,
        "question": question,
        "expected_answer": expected_answer,
        "model_answer": model_answer,
        "is_valid": validate_response(model_answer, ["right to life", "Article 26"])
    })

# Step 9: Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Step 10: Save the results to a new CSV file
output_path = r"C:\Users\Chris\Documents\ALEK\DataSci\Data\model_results.csv"
results_df.to_csv(output_path, index=False)

# Step 11: Display the results
print(results_df.head())

     chapter                                           question  \
0  Chapter 1                Who holds sovereign power in Kenya?   
1  Chapter 1  What is the significance of the supremacy of t...   
2  Chapter 1  Can Kenya be governed under any other law apar...   
3  Chapter 2                       What type of state is Kenya?   
4  Chapter 2  What are the national values and principles of...   

                                     expected_answer  \
0  Sovereign power belongs to the people of Kenya...   
1  The Constitution is the supreme law of Kenya. ...   
2  No, Kenya can only be governed under the Const...   
3  Kenya is a sovereign, democratic, and unitary ...   
4  They include patriotism, rule of law, democrac...   

                                model_answer  is_valid  
0  I don't have an answer for that question.     False  
1  I don't have an answer for that question.     False  
2  I don't have an answer for that question.     False  
3  I don't have an answer for th